# 🌍 TWS Forecasting Challenge – Starter Notebook (flat CSV release)

This notebook provides a simple baseline pipeline for predicting next-month Total Water
Storage (TWS).

**Updated for the flat CSV data release.** The competition data is now distributed as flat,
one-row-per-sample CSV files (`Train.csv`, `Test.csv`) instead
of gridded NetCDF files, so this notebook no longer needs `xarray` or any grid-indexing
logic — every feature is already a column.

One more thing that's different from earlier releases: roughly two-thirds of `TWS_t` values
in the test set are intentionally masked (`NaN`). This is not missing data — it's there to
prevent `TWS_t` at one test row from revealing the hidden target of another test row at the
same location the month before. See the `TWS_t_masked` column. Your model needs to handle
these missing values.

We will:
- Explore the data
- Visualize spatial and temporal patterns
- Build a simple baseline model
- Generate a submission

The goal is to improve over a strong persistence baseline.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
DATA_DIR = Path("../data")  # change if needed


TRAIN_FEATURES_PATH = DATA_DIR / "Train.csv"
TEST_FEATURES_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "SampleSubmission.csv"

In [ ]:
## Checking the file sizes
for file_path in [TRAIN_FEATURES_PATH, TEST_FEATURES_PATH, SAMPLE_SUBMISSION_PATH]:
    file_size = file_path.stat().st_size
    file_size_mb = file_size / (1024 * 1024)
    print(f"{file_path.name}: {file_size_mb:.2f} MB")


## 🔍 Dataset Overview

We are working with flat, tabular monthly data. Each row is one `(time, lat, lon)` sample,
identified by `ID` (format `YYYYMMDD_lat_lon`).

Available columns:
- `TWS_t` — current-month Total Water Storage( **masked in ~66%
  of test rows**, see `TWS_t_masked`)
- `SPEI_01_t`, `SPEI_03_t`, `SPEI_06_t`, `SPEI_12_t` — drought/wetness indices at multiple
  timescales
- `SOIL_MOISTURE_t` — current-month soil moisture
- `month_sin`, `month_cos` — cyclical month-of-year encoding
- `target` — TWS one month ahead (train only; this is what you're predicting)
- `TWS_t_masked` — boolean flag, test only, `True` where `TWS_t` has been masked


In [ ]:
def normalise_id_column(df: pd.DataFrame) -> pd.DataFrame:
    """Some releases use 'sample_id' instead of 'ID' for the test file — normalise it."""
    if "sample_id" in df.columns and "ID" not in df.columns:
        out = df.copy()
        out = out.rename(columns={"sample_id": "ID"})
        return out
    return df


train_features = normalise_id_column(pd.read_csv(TRAIN_FEATURES_PATH, parse_dates=["time"]))
test_features = normalise_id_column(pd.read_csv(TEST_FEATURES_PATH, parse_dates=["time"]))
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train_features:", train_features.shape)
print("test_features: ", test_features.shape)
print("sample_submission:", sample_submission_df.shape)

print("\nTrain columns:", train_features.columns.tolist())
print("Test columns: ", test_features.columns.tolist())


In [ ]:
train_features.head()


In [ ]:
print("Test TWS_t masking:")
print(test_features["TWS_t_masked"].value_counts())
print(f"Masked share: {test_features['TWS_t_masked'].mean():.1%}")


In [ ]:
test_features.head()

In [ ]:
available_train = set(train_features.columns)
available_test = set(test_features.columns)
available_common = sorted(
    (available_train & available_test) - {"ID", "time", "lat", "lon", "TWS_t_masked"}
)

print("Available common feature columns:")
print(available_common)


## 🌍 Spatial Patterns

Let's visualize the spatial distribution of Total Water Storage (TWS) for a single month.
This helps us understand:
- Large-scale geographic structure
- Differences between regions (e.g. land vs ocean, wet vs dry zones)


In [ ]:
def pivot_grid(df: pd.DataFrame, time_value, value_col: str) -> pd.DataFrame:
    """Pivot a single time slice of the flat dataframe into a (lat x lon) grid for plotting."""
    sub = df.loc[df["time"] == time_value, ["lat", "lon", value_col]]
    return sub.pivot(index="lat", columns="lon", values=value_col).sort_index().sort_index(axis=1)


t0 = train_features["time"].min()
print("Plotting time slice:", t0)


In [ ]:
grid_tws = pivot_grid(train_features, t0, "TWS_t")

plt.figure(figsize=(10, 5))
plt.imshow(grid_tws.values, cmap="RdBu", origin="lower",
           extent=[grid_tws.columns.min(), grid_tws.columns.max(),
                   grid_tws.index.min(), grid_tws.index.max()])
plt.colorbar(label="TWS(t)")
plt.title(f"TWS(t) at time {str(t0)[:10]}")
plt.xlabel("lon")
plt.ylabel("lat")
plt.show()


## 🌧️ Climate Forcing (SPEI)

We compare TWS with SPEI (Standardized Precipitation-Evapotranspiration Index).

SPEI represents drought/wetness conditions:
- Positive → wetter than normal
- Negative → drier than normal

This gives insight into how climate affects water storage.


In [ ]:
grid_spei = pivot_grid(train_features, t0, "SPEI_01_t")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

im0 = axs[0].imshow(grid_tws.values, cmap="RdBu", origin="lower")
axs[0].set_title("TWS(t)")
plt.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].imshow(grid_spei.values, cmap="BrBG", origin="lower")
axs[1].set_title("SPEI_01(t)")
plt.colorbar(im1, ax=axs[1], fraction=0.046)

plt.show()


## 📈 Temporal Dynamics

We inspect a time series at a single location.

Key observation:
- TWS has strong temporal memory
- Changes are gradual rather than abrupt

This suggests that **past TWS is a strong predictor of future TWS**.


In [ ]:
lat0 = 10.5
lon0 = 0.5

# Grid is on fixed 1-degree centres; snap the requested point to the nearest grid cell.
nearest_lat = train_features["lat"].sub(lat0).abs().idxmin()
nearest_lat = train_features.loc[nearest_lat, "lat"]
nearest_lon = train_features["lon"].sub(lon0).abs().idxmin()
nearest_lon = train_features.loc[nearest_lon, "lon"]

ts = train_features[(train_features["lat"] == nearest_lat) & (train_features["lon"] == nearest_lon)].sort_values("time")

plt.figure(figsize=(10, 4))
plt.plot(ts["time"], ts["TWS_t"], label="TWS(t)")
plt.plot(ts["time"], ts["SPEI_01_t"], label="SPEI_01(t)")
plt.legend()
plt.title(f"Time series at lat={nearest_lat}, lon={nearest_lon}")
plt.grid(True)
plt.show()


## 🔥 Persistence Baseline

A very strong baseline is:

TWS(t+1) ≈ TWS(t)

Since `Target` in the training file already **is** TWS at `t+1` (by construction), we can
visualize this relationship directly from `TWS_t` vs `Target` — no grid-shifting needed.

Any useful model must improve over this baseline.


In [ ]:
if 'target' in train_features:
    train_features = train_features.rename(columns={'target': 'Target'})

In [ ]:
pairs = train_features[["TWS_t", "Target"]].dropna()
n_plot = min(50000, len(pairs))
sample = pairs.sample(n=n_plot, random_state=42)

plt.figure(figsize=(6, 6))
plt.scatter(sample["TWS_t"], sample["Target"], s=3, alpha=0.3)

mn, mx = sample["TWS_t"].min(), sample["TWS_t"].max()
plt.plot([mn, mx], [mn, mx], "--")

plt.xlabel("TWS(t)")
plt.ylabel("TWS(t+1) = Target")
plt.title("Persistence strength")
plt.grid(True)
plt.show()


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }


# Test.csv hides TWS_t on ~66.5% of rows; Train.csv hides none. These two helpers
# are what keep the fit and the score honest about that gap.
TWS_MASK_RATE = 0.665


def mask_tws(df: pd.DataFrame, rate: float = TWS_MASK_RATE, seed: int = 0) -> pd.DataFrame:
    """Return a copy of `df` with TWS_t set to NaN on a random `rate` share of rows.

    Adds a TWS_t_masked flag so scores can be broken down by regime.
    """
    out = df.copy().reset_index(drop=True)
    masked = np.random.RandomState(seed).rand(len(out)) < rate
    out.loc[masked, "TWS_t"] = np.nan
    out["TWS_t_masked"] = masked
    return out


def augment_with_masking(df: pd.DataFrame, rate: float = TWS_MASK_RATE,
                         seed: int = 0) -> pd.DataFrame:
    """Original rows stacked with a masked copy.

    HistGradientBoostingRegressor only learns a split direction for NaN if it sees
    NaN while fitting. Train.csv never does, so without this the model has no
    learned behaviour for the two-thirds of test rows where TWS_t is hidden.
    """
    return pd.concat([df, mask_tws(df, rate, seed)], ignore_index=True)


def make_model():
    # No SimpleImputer: HistGradientBoostingRegressor handles NaN natively and learns
    # which way to send missing values. Median-filling TWS_t would instead tell the
    # model that 66.5% of test rows sit at the average, which they don't.
    return Pipeline([
        ("gbr", HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=300,
            max_depth=8,
            min_samples_leaf=50,
            l2_regularization=1.0,
            random_state=42
        ))
    ])


def build_xy(df: pd.DataFrame, feature_names: list, use_latlon: bool = False):
    """Flat-file equivalent of the old NetCDF-indexing build_xy: features are already
    columns, so this is just a column selection.
    Lat/Lon should not be use as features!"""
    X = df[feature_names].to_numpy(dtype=np.float32)

    if use_latlon:
        latlon = df[["lat", "lon"]].to_numpy(dtype=np.float32)
        X = np.column_stack([X, latlon])

    # Corrected: changed "target" to "Target"
    y = df["Target"].to_numpy(dtype=np.float32) if "Target" in df.columns else None

    # persistence baseline = TWS_t
    tws_t = df["TWS_t"].to_numpy(dtype=np.float32)

    return X, y, tws_t


def plot_maps_from_predictions(df, y_true, y_pred, time_value):
    """
    Plot TWS true, TWS predicted, and error = pred - true for one time slice.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain columns: time, lat, lon
    y_true, y_pred : array-like, aligned with df's row order
    time_value : timestamp-like, one time step present in df["time"]
    """
    mask = df["time"] == time_value

    df_sub = df.loc[mask, ["time", "lat", "lon"]].copy().reset_index(drop=True)
    y_true_sub = np.asarray(y_true)[mask.values]
    y_pred_sub = np.asarray(y_pred)[mask.values]

    if len(df_sub) == 0:
        raise ValueError(f"No rows found for time={time_value}")

    lats = np.sort(df_sub["lat"].unique())
    lons = np.sort(df_sub["lon"].unique())

    lat_to_idx = {float(v): i for i, v in enumerate(lats)}
    lon_to_idx = {float(v): i for i, v in enumerate(lons)}

    H, W = len(lats), len(lons)

    true_grid = np.full((H, W), np.nan, dtype=float)
    pred_grid = np.full((H, W), np.nan, dtype=float)

    for i, row in df_sub.iterrows():
        lat_i = lat_to_idx[float(row["lat"])]
        lon_i = lon_to_idx[float(row["lon"])]

        true_grid[lat_i, lon_i] = y_true_sub[i]
        pred_grid[lat_i, lon_i] = y_pred_sub[i]

    error_grid = pred_grid - true_grid

    vmin = np.nanpercentile(y_true_sub, 1)
    vmax = np.nanpercentile(y_true_sub, 99)
    emax = np.nanpercentile(np.abs(error_grid[np.isfinite(error_grid)]), 99)

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    im0 = axs[0].imshow(true_grid, cmap="RdBu", vmin=vmin, vmax=vmax, origin="lower")
    axs[0].set_title("TWS True")

    im1 = axs[1].imshow(pred_grid, cmap="RdBu", vmin=vmin, vmax=vmax, origin="lower")
    axs[1].set_title("TWS Predicted")

    im2 = axs[2].imshow(error_grid, cmap="coolwarm", vmin=-emax, vmax=emax, origin="lower")
    axs[2].set_title("Error (Pred - True)")

    for ax in axs:
        ax.axis("off")

    plt.colorbar(im0, ax=axs[0], fraction=0.046)
    plt.colorbar(im1, ax=axs[1], fraction=0.046)
    plt.colorbar(im2, ax=axs[2], fraction=0.046)

    plt.suptitle(f"Maps at time {pd.Timestamp(time_value).strftime('%Y-%m-%d')}")
    plt.tight_layout()
    plt.show()

In [ ]:
BASE_FEATURES = [
    "TWS_t",
    "month_sin",
    "month_cos",
]

# Note: the _tp1 (future-looking) columns from the old NetCDF release are not present in
# the flat CSV files, so they're no longer part of the optional feature set.
OPTIONAL_FEATURES = [
    "SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t",
]

feature_names = BASE_FEATURES + [f for f in OPTIONAL_FEATURES if f in available_common]
print(feature_names)


In [ ]:
unique_times = np.sort(train_features["time"].unique())
split_idx = int(len(unique_times) * 0.8)

fit_times = unique_times[:split_idx]
val_times = unique_times[split_idx:]

fit_df = train_features[train_features["time"].isin(fit_times)].copy().reset_index(drop=True)
val_df = train_features[train_features["time"].isin(val_times)].copy().reset_index(drop=True)

print("Fit period:", fit_df["time"].min(), "->", fit_df["time"].max())
print("Val period:", val_df["time"].min(), "->", val_df["time"].max())
print("Fit samples:", len(fit_df))
print("Val samples:", len(val_df))


In [ ]:
fit_df.head()

In [ ]:
val_df.head()

### Matching the validation set to the test set

`Test.csv` hides `TWS_t` on 66.5% of rows; `Train.csv` hides it on none. Scoring on an
unmasked validation set therefore measures a problem the submission never faces — and
since persistence alone reaches R² ≈ 0.36, `TWS_t` is the feature that gap is built on.

Two changes fix it:

1. **Mask the validation set** at the same 66.5% rate, so the reported score reflects
   what the leaderboard will see.
2. **Augment the fit set** with a masked copy. `HistGradientBoostingRegressor` learns a
   direction for `NaN` only from `NaN` seen during fitting; training exclusively on
   complete rows leaves that path unlearned.

The median `SimpleImputer` is also gone from `make_model()` — it erased the distinction
between "TWS_t is missing" and "TWS_t is average" on two-thirds of the test set.


In [ ]:
## Lat/Lon not used as features
# Fit on original + masked copies so the NaN path is actually trained.
fit_aug = augment_with_masking(fit_df, seed=0)

# Score against the test-time distribution: ~66.5% of TWS_t hidden.
val_masked = mask_tws(val_df, seed=1)
val_is_masked = val_masked["TWS_t_masked"].to_numpy()

X_fit, y_fit, tws_fit = build_xy(fit_aug, feature_names, use_latlon=False)
X_val, y_val, tws_val = build_xy(val_masked, feature_names, use_latlon=False)

# Kept only to show what the unmasked split used to report.
X_val_full, _, tws_val_full = build_xy(val_df, feature_names, use_latlon=False)

print("fit rows (augmented):", X_fit.shape)
print("val rows:", X_val.shape, "| masked share:", f"{val_is_masked.mean():.1%}")


In [ ]:
model = make_model()
model.fit(X_fit, y_fit)

y_val_pred = model.predict(X_val)            # test-like: 66.5% of TWS_t hidden
y_val_pred_full = model.predict(X_val_full)  # optimistic: TWS_t always present

# Persistence needs TWS_t. Where it is masked, fall back to climatology: these are
# standardised anomalies, so 0.0 is the mean.
y_persist_full = tws_val_full
y_persist_masked = np.where(np.isnan(tws_val), 0.0, tws_val)

print("Model      - test-like (66.5% of TWS_t masked)  <- the number to trust")
print(compute_metrics(y_val, y_val_pred))

print("\nModel      - TWS_t always present (optimistic, not the submission setting)")
print(compute_metrics(y_val, y_val_pred_full))

print("\nPersistence - TWS_t always present")
print(compute_metrics(y_val, y_persist_full))

print("\nPersistence - test-like, climatology fallback where masked")
print(compute_metrics(y_val, y_persist_masked))

m_model = compute_metrics(y_val, y_val_pred)
m_persist = compute_metrics(y_val, y_persist_masked)
print("\nImprovement over persistence (both test-like)")
print({
    "dMAE": m_persist["mae"] - m_model["mae"],
    "dRMSE": m_persist["rmse"] - m_model["rmse"],
    "dR2": m_model["r2"] - m_persist["r2"],
})

print("\nTest-like model score split by regime")
for label, sel in [("TWS_t present", ~val_is_masked), ("TWS_t masked", val_is_masked)]:
    print(f"  {label:14s} n={sel.sum():>7,}  {compute_metrics(y_val[sel], y_val_pred[sel])}")


In [ ]:
plt.figure(figsize=(6, 6))
n_plot = min(50000, len(y_val))
idx = np.random.RandomState(42).choice(len(y_val), size=n_plot, replace=False)

plt.scatter(y_val[idx], y_val_pred[idx], s=3, alpha=0.3)
mn = np.nanmin([y_val[idx].min(), y_val_pred[idx].min()])
mx = np.nanmax([y_val[idx].max(), y_val_pred[idx].max()])
plt.plot([mn, mx], [mn, mx], "--")
plt.xlabel("True TWS")
plt.ylabel("Predicted TWS")
plt.title("Validation predictions")
plt.grid(True)
plt.show()


In [ ]:
unique_val_times = np.sort(val_df["time"].unique())
time_example = unique_val_times[min(7, len(unique_val_times) - 1)]

plot_maps_from_predictions(
    val_df,
    y_val,
    y_val_pred,
    time_example
)


## 🚀 Fit the final model on all training data

Now refit on the full training set before predicting on the real test set.


In [ ]:
# Same masking augmentation as validation, so the submitted model has seen NaN TWS_t.
train_aug = augment_with_masking(train_features, seed=0)

X_train_full, y_train_full, tws_train_full = build_xy(
    train_aug,
    feature_names,
    use_latlon=False
)

print(X_train_full.shape, y_train_full.shape)

final_model = make_model()
final_model.fit(X_train_full, y_train_full)


## 📤 Build the submission

The flat test file already **is** the correct sample index (one row per official test
sample) — unlike the old NetCDF release, there's no need to reconstruct it from a global
grid mask. We just align it to `SampleSubmission.csv` by ID and predict directly.

`TWS_t` is `NaN` for the masked test rows. It is passed through as `NaN` on purpose:
`HistGradientBoostingRegressor` routes missing values down a learned branch, and the
masking augmentation above is what taught it that branch.


In [ ]:
sample_submission_df.head()


In [ ]:
test_features.head()

In [ ]:
### Really not necessary!
test_submission_df = sample_submission_df.merge(
    test_features,
    on="ID",
    how="left",
    validate="one_to_one"
)

print(test_submission_df.shape)
print(test_submission_df.isna().sum())
test_submission_df.head()


In [ ]:
X_test_submit, _, _ = build_xy(test_submission_df, feature_names, use_latlon=False)

y_test_pred = final_model.predict(X_test_submit)

submission_df = sample_submission_df.copy()
submission_df["Target"] = y_test_pred.astype(np.float32)
submission_df.columns = ["ID", "Target"]

submission_df.head()


In [ ]:
SUBMISSION_PATH = DATA_DIR / "Benchmark.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH)
